In [1]:
import os
from pathlib import Path

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
env_path = Path("../.env")
if env_path.exists():
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("\"'")
        os.environ.setdefault(key, value)

hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in ../.env or environment before running notebook.")

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, PeftConfig
BASE_MODEL = "Qwen/Qwen3-0.6B"
OUTPUT_DIR = "../outputs/sft_qwen3_0.6b"
DATASET_DIR = "../data/alpaca_gpt4"
ADAPTER_DIR = f"{OUTPUT_DIR}/final"
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="cuda:1")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [3]:
from datasets import load_dataset

tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    
ds = load_dataset(DATASET_DIR, split="validation")
def to_messages(example):
    user_text = example['instruction'].strip()
    if example['input'].strip():
        user_text += "\n\n" + example['input'].strip()
    return {
        "messages":[
            {"role": "user", "content": user_text},
            # {"role": "assistant", "content": example['output']},
        ]
    }

ds_messages = ds.map(to_messages, remove_columns=ds.column_names)

def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=True,

    )
    return {'text':text}
 
texts = ds_messages.map(apply_chat_template)

enc = tokenizer(
    [x for x in texts['text']],
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=256,
).to("cuda:1")

In [4]:
enc.input_ids

tensor([[151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198],
        ...,
        [151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198],
        [151643, 151643, 151643,  ..., 151644,  77091,    198]],
       device='cuda:1')

In [5]:
from transformers import GenerationConfig
import torch

torch.manual_seed(42)

    
gen_cfg_eval = GenerationConfig(
    do_sample=False,          # 贪心，稳定
    temperature=1.0,          # do_sample=False 时基本无效
    top_p=1.0,                # 同上
    top_k=0,                  # 同上
    max_new_tokens=256,       # 按任务调：128/256/512
    min_new_tokens=1,
    repetition_penalty=1.05,  # 轻微抑制复读
    no_repeat_ngram_size=0,   # 中文任务通常先不开
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

# gen_cfg_sample = GenerationConfig(
#     do_sample=True,
#     temperature=0.7,          # 常用 0.6~0.8
#     top_p=0.9,                # 常用 0.85~0.95
#     top_k=40,                 # 可设 40/50
#     max_new_tokens=256,
#     repetition_penalty=1.1,
#     eos_token_id=tokenizer.eos_token_id,
#     pad_token_id=tokenizer.pad_token_id,
# )

with torch.no_grad():
    outputs = base_model.generate_batch(
        inputs=enc.input_ids.tolist(),
        generation_config=gen_cfg_eval,
    )
decoded_outputs = [{'id': item.request_id, 'prompt': tokenizer.decode(item.prompt_ids, skip_special_tokens=True), 'generated': tokenizer.decode(item.generated_tokens, skip_special_tokens=True)} for item in outputs.values()]
import json

jsonl_path = f"../data/sft_eval/outputs_base.jsonl"


with open(jsonl_path, "w", encoding="utf-8") as f:
    for item in decoded_outputs:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")



print(f"已导出到: {jsonl_path}，共 {len(decoded_outputs)} 条")



The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Solving 1041 requests: 100%|██████████| 1041/1041 [08:26<00:00,  2.05request/s]


已导出到: ../data/sft_eval/outputs_base.jsonl，共 1041 条


In [ ]:
peft_config = PeftConfig.from_pretrained(ADAPTER_DIR)
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, device_map="cuda:1")

with torch.no_grad():
    outputs_peft = peft_model.generate_batch(
        inputs=enc.input_ids.tolist(),
        generation_config=gen_cfg_eval,
    )
decoded_outputs_peft = [{'id': item.request_id, 'prompt': tokenizer.decode(item.prompt_ids, skip_special_tokens=True), 'generated': tokenizer.decode(item.generated_tokens, skip_special_tokens=True)} for item in outputs_peft.values()]

jsonl_path_peft = f"../data/sft_eval/outputs_peft.jsonl"
with open(jsonl_path_peft, "w", encoding="utf-8") as f:
    for item in decoded_outputs_peft:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print(f"已导出到: {jsonl_path_peft}，共 {len(decoded_outputs_peft)} 条")